# Sinhala QA Evaluation (v6-matched)

Standalone evaluation of the merged v6 QA models (`isji/sinllama-3b-qa-v6-merged`,
`isji/sinllama-1b-qa-v6-merged`) on the external test split.

## Why the previous `qa-evaluation.ipynb` scored far below the v6 training run

The old notebook did not reproduce the inference path the model was trained for. Five
independent mismatches, each of which lowers the score on its own:

| | v6 training/eval (`qa-finetuning_v6.ipynb`) | old `qa-evaluation.ipynb` |
|---|---|---|
| Instruction text | `උපදෙස්: පහත සන්දර්භය පමණක් භාවිතා කර...` (4 bullet rules) | a different, shorter instruction |
| Context label | `සන්දර්භය:\n{context}` | `තොරතුරු (Context): {context}` |
| Answer cue | `පිළිතුර:\n` — answer, then EOS | `පිළිතුර: [` — answer expected inside `[...]` |
| Retrieval | answer-aware evidence windows, top-2 | none, raw context |
| Grounding gate | ungrounded generations → refusal | none |
| Decoding | `repetition_penalty=1.05`, 48 new tokens | `1.15`, 80 new tokens |
| Metrics | normalized EM (casefold + punctuation strip), token F1 with digit guard | raw EM, keyword soft-match |

The `පිළිتුර: [` bracket cue is the most damaging: the model was fine-tuned with
completion-only loss to emit the answer immediately after `පිළිතුර:\n` and then stop. Being
fed an unseen `[` continuation pushes it off-distribution, and the extractor then tries to
cut at a `]` the model has no reason to produce. On top of that, the old notebook dropped
the grounding gate — which is what produced v6's 100% unanswerable accuracy — and used a
stricter, non-casefolding normalizer, so even correct answers failed exact match.

**This notebook reproduces the v6 inference path exactly**, so the numbers it reports are
directly comparable to `qa-finetuning_v6.ipynb` and to
`llama_model_answers/{1B,3B}-v6-results.txt`. Nothing here is tuned to inflate scores; it
is the same code path the model was trained and originally measured under.

Cell 10 is an optional A/B diagnostic that runs both prompt formats on the same rows, so
the cause above is demonstrated on your own model rather than taken on trust.

In [ ]:
%uv pip install -q "transformers>=4.51,<5" accelerate safetensors huggingface_hub hf_transfer

In [ ]:
import json
import os
import re
import unicodedata
from collections import Counter
from pathlib import Path

import torch

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

MODEL_ID = "isji/sinllama-3b-qa-v6-merged"   # or "isji/sinllama-1b-qa-v6-merged"

TEST_CANDIDATES = [
    Path("/tmp/test_updated.jsonl"),
    Path("/tmp/test.jsonl"),
    Path("new_split_v2/test.jsonl"),
    Path("../new_split_v2/test.jsonl"),
]
TEST_PATH = next((p for p in TEST_CANDIDATES if p.is_file()), TEST_CANDIDATES[0])

_slug = re.sub(r"[^a-z0-9]+", "-", MODEL_ID.lower()).strip("-")
RESULTS_JSONL = Path(f"/tmp/{_slug}-eval.jsonl")
RESULTS_TXT = Path(f"/tmp/{_slug}-results.txt")

# ---- These must match qa-finetuning_v6.ipynb exactly ----
NO_ANSWER = "මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත."
MAX_LENGTH = 2048
MAX_NEW_TOKENS = 48          # audited: covers 100% of gold answers under this tokenizer
REPETITION_PENALTY = 1.05
TOP_K_WINDOWS = 2
GROUNDING_THRESHOLD = 0.50
USE_GROUNDING = True

RUN_AB_DIAGNOSTIC = True     # cell 10: compare old vs v6 prompt on a sample
AB_SAMPLE_SIZE = 20

print("Model     :", MODEL_ID)
print("Test file :", TEST_PATH, "| exists:", TEST_PATH.is_file())
print("Grounding :", f"on (threshold={GROUNDING_THRESHOLD}, top_k={TOP_K_WINDOWS})" if USE_GROUNDING else "off")

In [ ]:
# Only needed if the merged repo is private. Reads HF_TOKEN from the environment — never
# paste a token into this notebook; a committed token is a leaked credential.
from huggingface_hub import login

_token = os.environ.get("HF_TOKEN")
if _token:
    login(token=_token, add_to_git_credential=False)
    print("Logged in to Hugging Face from HF_TOKEN.")
else:
    print("No HF_TOKEN in the environment — continuing anonymously (fine for public repos).")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading merged model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=model_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
model.config.pad_token_id = tokenizer.pad_token_id

print("Loaded.  vocab:", len(tokenizer), "| dtype:", model_dtype, "| device:", model.device)
print("bos/eos/pad:", tokenizer.bos_token_id, "/", tokenizer.eos_token_id, "/", tokenizer.pad_token_id)

In [ ]:
# ---- Text handling, prompt, and data loading: copied from qa-finetuning_v6.ipynb ----

INSTRUCTION = f"""උපදෙස්: පහත සන්දර්භය පමණක් භාවිතා කර ප්‍රශ්නයට පිළිතුරු දෙන්න.
- පිළිතුර සන්දර්භයේ තිබේ නම්, එයින් කෙටිම නිශ්චිත වචන පෙළ පමණක් දෙන්න.
- අමතර පැහැදිලි කිරීම්, පිටත දැනුම හෝ අනුමාන එකතු නොකරන්න.
- සන්දර්භය ප්‍රශ්නයට අදාළ නොවේ නම්, ප්‍රශ්නයට පිළිතුරු දීමට සුදුසු නොවේ නම්, හෝ පිළිතුර සන්දර්භයේ පැහැදිලිව නොමැති නම්, හරියටම මෙය පමණක් දෙන්න: {NO_ANSWER}"""

SINHALA_WORD_RE = re.compile(r"[\w඀-෿]+", re.UNICODE)
STOPWORDS = {
    "හා", "සහ", "හෝ", "දී", "ද", "ය", "යි", "වේ", "විය", "වූ", "ලෙස",
    "විසින්", "සඳහා", "සිට", "දක්වා", "එම", "මෙම", "ඒ", "ඔහු", "ඇය",
    "කුමක්ද", "කවුද", "කවදාද", "කෙසේද", "කොපමණද", "මොනවාද",
}


def clean_text(value):
    text = unicodedata.normalize("NFC", str(value or ""))
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()


def lexical_tokens(value):
    tokens = [token.casefold() for token in SINHALA_WORD_RE.findall(clean_text(value))]
    return [token for token in tokens if len(token) >= 2 and token not in STOPWORDS]


def build_prompt(context, question):
    return (
        f"{INSTRUCTION}\n\n"
        f"සන්දර්භය:\n{clean_text(context)}\n\n"
        f"ප්‍රශ්නය:\n{clean_text(question)}\n\n"
        "පිළිතුර:\n"
    )


def canonical_answer(item):
    if item.get("answerable") is False:
        return NO_ANSWER
    answer = clean_text(item.get("answer", ""))
    return answer if answer else NO_ANSWER


def load_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"Required JSONL file not found: {path}")

    records = []
    fingerprints = set()
    dropped = 0
    duplicates = 0

    with path.open("r", encoding="utf-8-sig") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}:{line_number}: {error}") from error

            question = clean_text(item.get("question"))
            context = clean_text(item.get("context"))
            answerable = item.get("answerable")
            if type(answerable) is not bool:
                answerable = bool(clean_text(item.get("answer")))

            normalized = {
                "question": question,
                "context": context,
                "answer": clean_text(item.get("answer")),
                "answerable": answerable,
                "grade": item.get("grade"),
                "chapter": item.get("chapter"),
            }
            if not question or not context or (answerable and not normalized["answer"]):
                dropped += 1
                continue

            fingerprint = (
                normalized["question"],
                normalized["context"],
                normalized["answer"],
                normalized["answerable"],
            )
            if fingerprint in fingerprints:
                duplicates += 1
                continue
            fingerprints.add(fingerprint)
            records.append(normalized)

    return records, dropped, duplicates


print("Prompt and data helpers ready (v6-matched).")

In [ ]:
# ---- Evidence-window retrieval and grounded inference: copied from qa-finetuning_v6.ipynb ----

def token_supported(token, normalized_context):
    if token in normalized_context:
        return True
    # Sinhala case endings often add one character; a short stem check preserves grounded variants.
    return len(token) >= 4 and token[:-1] in normalized_context


def rank_context_windows(context, query, token_budget, top_k=1):
    context_ids = tokenizer(context, add_special_tokens=False)["input_ids"]
    if len(context_ids) <= token_budget:
        return [(context, 1.0)]

    query_tokens = set(lexical_tokens(query))
    stride = max(64, token_budget // 2)
    candidates = []

    for start in range(0, len(context_ids), stride):
        chunk_ids = context_ids[start : start + token_budget]
        if len(chunk_ids) < 32:
            continue
        chunk = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ).strip()
        normalized_chunk = " ".join(lexical_tokens(chunk))
        if query_tokens:
            matched = sum(token_supported(token, normalized_chunk) for token in query_tokens)
            score = matched / len(query_tokens)
        else:
            score = 0.0
        candidates.append((chunk, score, start))
        if start + token_budget >= len(context_ids):
            break

    candidates.sort(key=lambda value: (-value[1], value[2]))
    return [(chunk, score) for chunk, score, _ in candidates[:top_k]]


def context_budget(question, completion):
    fixed_tokens = len(tokenizer(build_prompt("", question), add_special_tokens=False)["input_ids"])
    completion_tokens = len(tokenizer(completion, add_special_tokens=False)["input_ids"])
    return max(128, MAX_LENGTH - fixed_tokens - completion_tokens - 24)


def evidence_support(answer, context):
    answer_tokens = lexical_tokens(answer)
    if not answer_tokens:
        return 0.0
    normalized_context = " ".join(lexical_tokens(context))
    supported = sum(token_supported(token, normalized_context) for token in answer_tokens)
    return supported / len(answer_tokens)


def generate_candidate(context_window, question):
    prompt = build_prompt(context_window, question)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=REPETITION_PENALTY,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[-1] :]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    answer = answer.splitlines()[0].strip(" []{}()<>\"'`") if answer else ""
    return answer


def run_qa(context, question, use_grounding=USE_GROUNDING):
    completion_stub = NO_ANSWER + (tokenizer.eos_token or "")
    budget = context_budget(question, completion_stub)
    windows = rank_context_windows(context, question, budget, top_k=TOP_K_WINDOWS)
    candidates = []

    for window, retrieval_score in windows:
        raw_answer = generate_candidate(window, question)
        support = 1.0 if is_no_answer(raw_answer) else evidence_support(raw_answer, window)
        candidates.append({
            "raw_answer": raw_answer,
            "window": window,
            "retrieval_score": retrieval_score,
            "support": support,
        })

    grounded = [
        candidate for candidate in candidates
        if candidate["raw_answer"] and not is_no_answer(candidate["raw_answer"])
        and candidate["support"] >= GROUNDING_THRESHOLD
    ]

    if grounded:
        best = max(
            grounded,
            key=lambda candidate: (candidate["support"], candidate["retrieval_score"]),
        )
        final_answer = best["raw_answer"]
    else:
        best = max(candidates, key=lambda candidate: candidate["retrieval_score"])
        final_answer = NO_ANSWER if use_grounding else best["raw_answer"]

    return {
        "answer": final_answer,
        "raw_answer": best["raw_answer"],
        "support": best["support"],
        "retrieval_score": best["retrieval_score"],
        "candidate_count": len(candidates),
    }


print("Grounded run_qa(context, question) ready (v6-matched).")

In [ ]:
# ---- Metrics: copied from qa-finetuning_v6.ipynb ----

def normalize_answer(value):
    text = clean_text(value).casefold()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" \t\r\n[]{}()<>\"'`.,!?;:।෴")


def is_no_answer(value):
    normalized = normalize_answer(value)
    return normalized == normalize_answer(NO_ANSWER) or "ප්‍රමාණවත් තොරතුරු නොමැත" in normalized


def token_f1(prediction, reference):
    prediction_digits = re.findall(r"\d+", normalize_answer(prediction))
    reference_digits = re.findall(r"\d+", normalize_answer(reference))
    # A wrong year is a wrong answer even when the surrounding words overlap.
    if reference_digits and prediction_digits != reference_digits:
        return 0.0
    prediction_tokens = lexical_tokens(prediction)
    reference_tokens = lexical_tokens(reference)
    if not prediction_tokens and not reference_tokens:
        return 1.0
    if not prediction_tokens or not reference_tokens:
        return 0.0
    common = Counter(prediction_tokens) & Counter(reference_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(prediction_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)


def keyword_match(prediction, gold):
    """Legacy soft match from the old notebook, reported for continuity only."""
    pred, target = normalize_answer(prediction), normalize_answer(gold)
    if not pred or not target:
        return False
    if pred == target or target in pred or pred in target:
        return True
    target_tokens = [t for t in target.split() if len(t) > 1]
    if not target_tokens:
        return False
    return sum(1 for t in target_tokens if t in pred) / len(target_tokens) >= 0.6


print("Scoring helpers ready (v6-matched).")

In [ ]:
test_records, dropped_rows, duplicate_rows = load_jsonl(TEST_PATH)
LOAD_HEADER = [
    f"Loaded {len(test_records)} unique records from {TEST_PATH}",
    f"Dropped invalid/empty: {dropped_rows}; exact duplicates removed: {duplicate_rows}",
]
for line in LOAD_HEADER:
    print(line)
print(Counter(r["answerable"] for r in test_records))

gold_lengths = sorted(
    len(tokenizer(canonical_answer(r), add_special_tokens=False)["input_ids"]) for r in test_records
)
print(
    f"\nGold answer tokens: median {gold_lengths[len(gold_lengths)//2]}, "
    f"p95 {gold_lengths[int(len(gold_lengths)*0.95)]}, max {gold_lengths[-1]} "
    f"(MAX_NEW_TOKENS={MAX_NEW_TOKENS})"
)
if gold_lengths[-1] > MAX_NEW_TOKENS:
    print("WARNING: raise MAX_NEW_TOKENS — the longest gold answer does not fit.")

In [ ]:
# ---- Prompt-format check + smoke test ----
row = test_records[0]
preview = build_prompt(row["context"], row["question"])

# Guard against silently drifting away from the trained format again.
assert preview.startswith("උපදෙස්: පහත සන්දර්භය පමණක්"), "instruction text does not match v6"
assert "\nසන්දර්භය:\n" in preview, "context label does not match v6"
assert "\nප්‍රශ්නය:\n" in preview, "question label does not match v6"
assert preview.endswith("පිළිතුර:\n"), "answer cue does not match v6 (must NOT end with '[')"
print("Prompt format matches the v6 training format.\n")
print("--- rendered prompt ---")
print(preview)

result = run_qa(row["context"], row["question"])
print("--- smoke test ---")
print("Question :", row["question"])
print("Reference:", canonical_answer(row))
print("Raw      :", result["raw_answer"])
print("Final    :", result["answer"])
print("Support  :", f"{result['support']:.3f}")

In [ ]:
# ---- Optional A/B diagnostic: old prompt vs v6 prompt on the same rows ----
# Demonstrates on your own model that the prompt format (not the model) caused the drop.
if RUN_AB_DIAGNOSTIC:
    def build_legacy_prompt(context, question):
        return (
            "උපදෙස්: ඔබ සිංහල ප්‍රශ්න-පිළිතුරු සහායකයෙකි. පහත සපයා ඇති තොරතුරු (Context) පමණක් භාවිතා කරමින් "
            "ප්‍රශ්නයට නිවැරදි, සෘජු, කෙටි පිළිතුරක් දෙන්න. Context තුළ පිළිතුර නොමැති නම් හෝ ප්‍රමාණවත් තොරතුරු නොමැති නම්, "
            f"“{NO_ANSWER}” යනුවෙන් පමණක් පිළිතුරු දෙන්න. Context වලින් පිටත දැනුම, අනුමාන, හෝ අමතර විස්තර භාවිතා නොකරන්න.\n\n"
            f"තොරතුරු (Context): {context}\n"
            f"ප්‍රශ්නය: {question}\n"
            "පිළිතුර: ["
        )

    def run_qa_legacy(context, question):
        prompt = build_legacy_prompt(context, question)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=80,
                do_sample=False,
                repetition_penalty=1.15,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )
        full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        answer = full_text.split("පිළිතුර: [")[-1].strip()
        if "]" in answer:
            answer = answer.split("]", 1)[0].strip()
        return answer

    sample = test_records[:AB_SAMPLE_SIZE]
    legacy_em = legacy_f1 = v6_em = v6_f1 = 0.0
    print(f"Comparing prompt formats on {len(sample)} rows...\n")
    for item in sample:
        reference = canonical_answer(item)
        legacy_pred = run_qa_legacy(item["context"], item["question"])
        v6_pred = run_qa(item["context"], item["question"])["answer"]
        legacy_em += normalize_answer(legacy_pred) == normalize_answer(reference)
        v6_em += normalize_answer(v6_pred) == normalize_answer(reference)
        legacy_f1 += token_f1(legacy_pred, reference)
        v6_f1 += token_f1(v6_pred, reference)

    n = len(sample)
    legacy_label = "old (පිළිතුර: [)"
    v6_label = "v6 (පිළිතුර: newline)"
    print(f"{'prompt format':24s} {'EM':>8s} {'token F1':>10s}")
    print(f"{legacy_label:24s} {100*legacy_em/n:7.1f}% {legacy_f1/n:10.4f}")
    print(f"{v6_label:24s} {100*v6_em/n:7.1f}% {v6_f1/n:10.4f}")
    print("\nIf the v6 row is materially higher, the old notebook's low scores were an "
          "evaluation artifact, not a regression in the model.")
else:
    print("A/B diagnostic skipped (RUN_AB_DIAGNOSTIC = False).")

In [ ]:
# ---- Full evaluation, reported exactly like qa-finetuning_v6.ipynb ----
predictions = []
transcript_lines = list(LOAD_HEADER)

exact_correct = raw_exact_correct = 0
f1_total = 0.0
answerable_correct = answerable_total = 0
unanswerable_correct = unanswerable_total = 0
unsupported_rejections = 0
predicted_no_answer_count = correct_no_answer_count = 0
soft_correct = 0

with RESULTS_JSONL.open("w", encoding="utf-8", newline="\n") as results_file:
    for index, item in enumerate(test_records, 1):
        reference = canonical_answer(item)
        result = run_qa(item["context"], item["question"], use_grounding=USE_GROUNDING)
        prediction = result["answer"]
        raw_prediction = result["raw_answer"]

        exact = normalize_answer(prediction) == normalize_answer(reference)
        raw_exact = normalize_answer(raw_prediction) == normalize_answer(reference)
        f1 = token_f1(prediction, reference)
        predicted_no_answer = is_no_answer(prediction)

        exact_correct += int(exact)
        raw_exact_correct += int(raw_exact)
        f1_total += f1
        soft_correct += int(keyword_match(prediction, reference))
        predicted_no_answer_count += int(predicted_no_answer)
        correct_no_answer_count += int(predicted_no_answer and not item["answerable"])
        unsupported_rejections += int(
            prediction == NO_ANSWER and raw_prediction and not is_no_answer(raw_prediction)
        )

        if item["answerable"]:
            answerable_total += 1
            answerable_correct += int(exact)
        else:
            unanswerable_total += 1
            unanswerable_correct += int(exact)

        results_file.write(json.dumps({
            "index": index,
            "grade": item.get("grade"),
            "chapter": item.get("chapter"),
            "question": item["question"],
            "reference": reference,
            "raw_prediction": raw_prediction,
            "prediction": prediction,
            "answerable": item["answerable"],
            "exact_match": exact,
            "token_f1": f1,
            "evidence_support": result["support"],
            "retrieval_score": result["retrieval_score"],
        }, ensure_ascii=False) + "\n")
        results_file.flush()

        block = [
            "",
            "=" * 100,
            f"[{index}/{len(test_records)}]",
            f"Grade    : {item.get('grade')} | Chapter: {item.get('chapter')}",
            f"Answerable: {item['answerable']}",
            f"Question : {item['question']}",
            f"Reference: {reference}",
            f"Raw      : {raw_prediction}",
            f"Final    : {prediction}",
            f"Support  : {result['support']:.3f}",
            f"Exact/F1 : {exact} / {f1:.3f}",
        ]
        print("\n".join(block), flush=True)
        transcript_lines.extend(block)

print("\nFinished evaluation.")

In [ ]:
total = len(test_records)
false_answers = unanswerable_total - unanswerable_correct
no_answer_precision = correct_no_answer_count / max(predicted_no_answer_count, 1)
no_answer_recall = correct_no_answer_count / max(unanswerable_total, 1)
no_answer_f1 = (
    2 * no_answer_precision * no_answer_recall / (no_answer_precision + no_answer_recall)
    if no_answer_precision + no_answer_recall else 0.0
)

SUMMARY = [
    "",
    "=" * 100,
    "EXTERNAL TEST RESULTS",
    "=" * 100,
    f"Model                : {MODEL_ID}",
    f"Test file            : {TEST_PATH}",
    f"Grounding gate       : {'on' if USE_GROUNDING else 'off'} "
    f"(threshold={GROUNDING_THRESHOLD}, top_k={TOP_K_WINDOWS})",
    "",
    f"Grounded exact match : {exact_correct}/{total} ({100 * exact_correct / total:.2f}%)",
    f"Raw exact match      : {raw_exact_correct}/{total} ({100 * raw_exact_correct / total:.2f}%)",
    f"Soft match (legacy)  : {soft_correct}/{total} ({100 * soft_correct / total:.2f}%)",
    f"Mean token F1        : {f1_total / total:.4f}",
    f"Answerable exact     : {answerable_correct}/{answerable_total} "
    f"({100 * answerable_correct / max(answerable_total, 1):.2f}%)",
    f"Unanswerable exact   : {unanswerable_correct}/{unanswerable_total} "
    f"({100 * unanswerable_correct / max(unanswerable_total, 1):.2f}%)",
    f"False-answer rate on unanswerable: {false_answers}/{unanswerable_total} "
    f"({100 * false_answers / max(unanswerable_total, 1):.2f}%)",
    f"No-answer precision/recall/F1: {no_answer_precision:.4f} / {no_answer_recall:.4f} / "
    f"{no_answer_f1:.4f}",
    f"Unsupported generations rejected: {unsupported_rejections}",
    "=" * 100,
]

for line in SUMMARY:
    print(line)
transcript_lines.extend(SUMMARY)

with RESULTS_TXT.open("w", encoding="utf-8", newline="\n") as handle:
    handle.write("\n".join(transcript_lines) + "\n")

print(f"\nPer-item results : {RESULTS_JSONL}")
print(f"Transcript       : {RESULTS_TXT}")
print("\nThis transcript uses the same block format as llama_model_answers/*-v6-results.txt,")
print("so it can be diffed directly against the original v6 run.")

In [ ]:
# ---- Failure inspection ----
rows = [json.loads(l) for l in RESULTS_JSONL.open(encoding="utf-8") if l.strip()]

hallucinated = [r for r in rows if not r["answerable"] and not is_no_answer(r["prediction"])]
print(f"Fabricated answers on unanswerable questions: {len(hallucinated)}")
for r in hallucinated[:5]:
    print("-" * 90)
    print("Q   :", r["question"])
    print("Pred:", r["prediction"], f"| support {r['evidence_support']:.3f}")

low_f1 = [r for r in rows if r["answerable"] and r["token_f1"] < 0.5]
print(f"\nAnswerable rows below 0.5 token F1: {len(low_f1)}")
for r in low_f1[:10]:
    print("-" * 90)
    print("Q   :", r["question"])
    print("Ref :", r["reference"])
    print("Raw :", r["raw_prediction"])
    print("Pred:", r["prediction"], f"| F1 {r['token_f1']:.3f} | support {r['evidence_support']:.3f}")